# Section 4.1 — CNN from Scratch
## Cat vs Dog Binary Classification
**Image Processing Course — Spring 2026 | Elsewedy University of Technology**

This notebook builds a full CNN architecture from scratch, layer by layer:
- Convolutional Layers + ReLU Activation
- MaxPooling Layers
- Batch Normalization
- Flatten + Dense (Fully Connected) Layers
- Output Layer with Sigmoid activation (Binary Classification)

> **Prerequisites:** Run `preprocessing.ipynb` first to generate `data/split/` folders.

## 1. Import Libraries

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D,
    BatchNormalization,
    Flatten, Dense, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

## 2. Paths & Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────
BASE_DIR    = Path(r'D:\my_project')
SPLIT_DIR   = BASE_DIR / 'data' / 'split'
MODELS_DIR  = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-4

print(f'Train  : {SPLIT_DIR / "train"}')
print(f'Val    : {SPLIT_DIR / "val"}')
print(f'Test   : {SPLIT_DIR / "test"}')
print(f'Image size  : {IMG_SIZE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Max epochs  : {EPOCHS}')

## 3. Load Data with flow_from_directory
Images are loaded **in batches** (32 at a time) directly from folders.
This avoids loading all images into RAM at once — no memory errors.

In [ ]:
# Normalize pixel values to [0, 1]
datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen = datagen.flow_from_directory(
    SPLIT_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=42
)

val_gen = datagen.flow_from_directory(
    SPLIT_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = datagen.flow_from_directory(
    SPLIT_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False   # Must be False for correct evaluation
)

print(f'\nClass mapping : {train_gen.class_indices}')
print(f'Train samples : {train_gen.samples}')
print(f'Val samples   : {val_gen.samples}')
print(f'Test samples  : {test_gen.samples}')

## 4. Build CNN Architecture from Scratch

Architecture overview:
```
Input (224x224x3)
    │
    ├── Block 1: Conv2D(32) → ReLU → BatchNorm → MaxPooling → Dropout
    ├── Block 2: Conv2D(64) → ReLU → BatchNorm → MaxPooling → Dropout
    ├── Block 3: Conv2D(128)→ ReLU → BatchNorm → MaxPooling → Dropout
    │
    ├── Flatten
    ├── Dense(256) → ReLU → BatchNorm → Dropout
    └── Dense(1)   → Sigmoid  ← Output Layer
```

In [ ]:
model = Sequential(name='CNN_from_Scratch')

# ── Convolutional Block 1 ──────────────────────────────────
# Learn basic features: edges, colors
model.add(Conv2D(filters=32, kernel_size=(3, 3),
                 activation='relu', padding='same',
                 input_shape=(224, 224, 3)))
model.add(BatchNormalization())          # Stabilize & speed up training
model.add(MaxPooling2D(pool_size=(2, 2))) # Reduce spatial size: 224→112
model.add(Dropout(0.25))                 # Prevent overfitting

# ── Convolutional Block 2 ──────────────────────────────────
# Learn mid-level features: textures, patterns
model.add(Conv2D(filters=64, kernel_size=(3, 3),
                 activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2))) # 112→56
model.add(Dropout(0.25))

# ── Convolutional Block 3 ──────────────────────────────────
# Learn high-level features: shapes, object parts
model.add(Conv2D(filters=128, kernel_size=(3, 3),
                 activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2))) # 56→28
model.add(Dropout(0.3))

# ── Flatten + Fully Connected Layers ──────────────────────
model.add(Flatten())                     # Convert 3D → 1D
model.add(Dense(units=256, activation='relu'))  # Fully Connected Layer
model.add(BatchNormalization())
model.add(Dropout(0.5))                  # Strong dropout before output

# ── Output Layer ───────────────────────────────────────────
# Sigmoid → outputs probability between 0 and 1
# 0 = Cat, 1 = Dog
model.add(Dense(units=1, activation='sigmoid'))

model.summary()

## 5. Compile the Model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',   # Standard loss for binary classification
    metrics=['accuracy']
)

total_params     = model.count_params()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)

print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Optimizer            : Adam (lr={LR})')
print(f'Loss function        : Binary Crossentropy')

## 6. Training Callbacks

In [ ]:
callbacks = [
    # Stop training if val_loss doesn't improve for 7 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate if stuck
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Save the best model automatically
    ModelCheckpoint(
        filepath=str(MODELS_DIR / 'cnn_scratch_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Callbacks ready:')
print('  EarlyStopping      → stops if no improvement for 7 epochs')
print('  ReduceLROnPlateau  → halves learning rate if stuck for 3 epochs')
print('  ModelCheckpoint    → saves best model to models/')

## 7. Train the Model

In [ ]:
print('Starting training...')
print(f'  Max epochs  : {EPOCHS}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Train steps : {train_gen.samples // BATCH_SIZE}')
print(f'  Val steps   : {val_gen.samples // BATCH_SIZE}')
print()

start_time = time.time()

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
epochs_trained = len(history.history['loss'])

print(f'\nTraining completed in {training_time:.1f}s ({training_time/60:.1f} min)')
print(f'Epochs trained      : {epochs_trained}')
print(f'Best val accuracy   : {max(history.history["val_accuracy"]):.4f}')

## 8. Training & Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN from Scratch — Training History', fontsize=14, fontweight='bold')

# Accuracy curve
axes[0].plot(history.history['accuracy'],     label='Train', color='#4ECDC4', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#FF6B6B', linewidth=2)
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curve
axes[1].plot(history.history['loss'],     label='Train', color='#4ECDC4', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#FF6B6B', linewidth=2)
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'cnn_scratch_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_scratch_training_curves.png')

## 9. Evaluate on Test Set

In [ ]:
# Get predictions
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1).flatten()
y_pred       = (y_pred_proba > 0.5).astype(int)
y_true       = test_gen.classes

test_loss, test_acc = model.evaluate(test_gen, verbose=0)

print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')

### 9.1 Classification Report (Accuracy, Precision, Recall, F1)

In [ ]:
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Cat', 'Dog']))

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)

print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')

### 9.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cat', 'Dog'],
            yticklabels=['Cat', 'Dog'],
            linewidths=0.5)
plt.title('Confusion Matrix — CNN from Scratch', fontweight='bold', fontsize=13)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'cnn_scratch_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_scratch_confusion_matrix.png')

### 9.3 ROC / AUC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#4ECDC4', lw=2.5,
         label=f'CNN from Scratch (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='#4ECDC4')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — CNN from Scratch', fontweight='bold', fontsize=13)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'cnn_scratch_roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC Score : {roc_auc:.4f}')
print('Saved: cnn_scratch_roc_curve.png')

### 9.4 Sample Predictions

In [ ]:
# Load a batch of test images for visualization
test_gen.reset()
sample_images, sample_labels = next(test_gen)
sample_preds_proba = model.predict(sample_images, verbose=0).flatten()
sample_preds       = (sample_preds_proba > 0.5).astype(int)

class_names = {0: 'Cat', 1: 'Dog'}

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Sample Predictions — CNN from Scratch\n(Green = Correct | Red = Wrong)',
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    ax.imshow(sample_images[i])
    true_label = class_names[int(sample_labels[i])]
    pred_label = class_names[int(sample_preds[i])]
    conf       = sample_preds_proba[i] if sample_preds[i] == 1 else 1 - sample_preds_proba[i]
    color      = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.0%})',
                 color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'cnn_scratch_sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_scratch_sample_predictions.png')

## 10. Save the Model

In [ ]:
model.save(str(MODELS_DIR / 'cnn_scratch_final.keras'))
print('Model saved to models/cnn_scratch_final.keras')

## 11. Final Results Summary

In [ ]:
print('=' * 55)
print('  CNN FROM SCRATCH — FINAL RESULTS')
print('=' * 55)
print(f'  Accuracy       : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision      : {prec:.4f}')
print(f'  Recall         : {rec:.4f}')
print(f'  F1-Score       : {f1:.4f}')
print(f'  AUC            : {roc_auc:.4f}')
print(f'  Test Loss      : {test_loss:.4f}')
print(f'  Epochs Trained : {epochs_trained}')
print(f'  Training Time  : {training_time:.1f}s ({training_time/60:.1f} min)')
print(f'  Total Params   : {total_params:,}')
print('=' * 55)
print()
print('Results saved to results/')
print('  cnn_scratch_training_curves.png')
print('  cnn_scratch_confusion_matrix.png')
print('  cnn_scratch_roc_curve.png')
print('  cnn_scratch_sample_predictions.png')
print()
print('Model saved to models/')
print('  cnn_scratch_final.keras')
print('  cnn_scratch_best.keras')